# Importancia de la topología informacional de BTC

Análisis causal y reproducible de la utilidad de `BTC_y` como coordenada del simplex transaccional y como descriptor de regímenes futuros de volatilidad.

El notebook reproduce las etapas A–K del análisis exploratorio: dependencia lineal y no lineal, lags, heterocedasticidad, volatilidad futura, regresión, clasificación de tres y cinco regímenes y contrastes anidados de utilidad informacional.

**Contrato:** sin `shuffle`, features disponibles hasta `t`, targets posteriores a `t`, umbrales ajustados sólo en train, frontera train/test purgada y walk-forward expansivo con `gap` temporal.

## 1. Dependencias

Colab ya incluye NumPy, pandas, SciPy, scikit-learn y Matplotlib. Se aseguran únicamente los lectores Parquet y LightGBM.

In [ ]:
%pip install -q lightgbm pyarrow

## 2. Drive y configuración

- Dejá `RUN_PIPELINE=True` para estimar todo.
- Cambialo a `False` para leer resultados ya guardados sin volver a entrenar.
- `RUN_WALK_FORWARD=False` permite una primera prueba rápida sólo con holdout.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/Neural/NPP/Cripto')
PRICE_PATH = Path('/content/drive/MyDrive/0626p.parquet')
SIMPLEX_PATH = Path('/content/drive/MyDrive/0626dfyp.parquet')
OUTPUT_DIR = PROJECT_DIR / 'topologia_informacional_BTC'
DRIVE_SCRIPT = PROJECT_DIR / 'analisis_topologia_informacional_btc.py'

RUN_PIPELINE = True
RUN_WALK_FORWARD = True
BACKEND = 'lightgbm'  # 'lightgbm' o 'histgb'
SEED = 42
FOLDS = 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Resultados:', OUTPUT_DIR)

## 3. Motor científico versionado

Se utiliza primero la copia guardada junto al notebook en Drive. Si no existe, se descarga la versión exacta del commit científico publicado en GitHub.

In [ ]:
import urllib.request

PINNED_COMMIT = 'd62d9f4fb4f0aa26bdb9f3b9a1b00681c28983e3'
RAW_SCRIPT_URL = (
    'https://raw.githubusercontent.com/Miguithub/NPP/'
    f'{PINNED_COMMIT}/analisis_topologia_informacional_btc.py'
)
RUNTIME_SCRIPT = Path('/content/analisis_topologia_informacional_btc.py')

if DRIVE_SCRIPT.exists():
    RUNTIME_SCRIPT.write_bytes(DRIVE_SCRIPT.read_bytes())
    print('Motor cargado desde Drive:', DRIVE_SCRIPT)
else:
    urllib.request.urlretrieve(RAW_SCRIPT_URL, RUNTIME_SCRIPT)
    print('Motor descargado desde el commit:', PINNED_COMMIT)

assert RUNTIME_SCRIPT.stat().st_size > 10_000, 'El script descargado está incompleto.'

## 4. Ejecución completa o reanudación desde CSV

Esta celda estima el pipeline sólo cuando `RUN_PIPELINE=True`. Los resultados quedan persistidos inmediatamente en `OUTPUT_DIR`; en otra sesión podés poner el interruptor en `False` y continuar directamente con la lectura.

In [ ]:
import subprocess
import sys

if RUN_PIPELINE:
    command = [
        sys.executable, str(RUNTIME_SCRIPT),
        '--price-path', str(PRICE_PATH),
        '--simplex-path', str(SIMPLEX_PATH),
        '--output-dir', str(OUTPUT_DIR),
        '--backend', BACKEND,
        '--seed', str(SEED),
        '--folds', str(FOLDS),
    ]
    if not RUN_WALK_FORWARD:
        command.append('--skip-walk-forward')
    subprocess.run(command, check=True)
else:
    required = OUTPUT_DIR / '09c_clasificacion_train.csv'
    if not required.exists():
        raise FileNotFoundError(
            f'No hay resultados guardados en {OUTPUT_DIR}. Activá RUN_PIPELINE.'
        )
    print('Se reutilizan los resultados existentes; no se reentrena.')

## 5. Funciones de lectura

Cada CSV contiene columnas `experiment`, `model` y `feature_set` para que ninguna métrica quede separada del modelo que la produjo.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

def load_result(name):
    path = OUTPUT_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    if path.stat().st_size <= 2:
        return pd.DataFrame()
    return pd.read_csv(path)

print('Archivos disponibles:')
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print('-', path.name)

## 6. Auditoría de datos y split

Verifica rango temporal, observaciones inválidas, tamaño de train/test, purga y ausencia de shuffle.

In [ ]:
audit = load_result('00_auditoria_datos.csv')
display(audit)

## 7. Etapas A–C — dependencia básica y tendencia compartida

Primero se comprueba si `BTC_y` se relaciona con el nivel de precio. La correlación parcial controla por el precio contemporáneo para detectar asociaciones generadas por persistencia o tendencia común.

In [ ]:
basic = load_result('01_dependencia_basica.csv')
derived = load_result('02_features_derivadas_vs_precio.csv')
display(basic)
display(derived.sort_values('mutual_information', ascending=False))

## 8. Etapas D–F — cambios, lags y naturaleza de la señal

Se distingue si `ΔBTC_y` informa dirección, magnitud, colas o varianza condicional. Una AUC cercana a 0.5 descarta dirección; una caída de MI al estandarizar por volatilidad apunta a información de régimen.

In [ ]:
relations = load_result('03_dBTC_y_vs_precio_retorno.csv')
lag_scan = load_result('04_barrido_lags_retorno.csv')
lag_decomposition = load_result('05_descomposicion_lag_4.csv')
display(relations)
display(lag_scan.sort_values('mutual_information', ascending=False).head(10))
display(lag_decomposition)

## 9. Etapa G — topología y volatilidad futura

El barrido lag × horizonte se calcula sólo sobre desarrollo. La estabilidad entre mitades permite detectar un pico aislado que no se repite temporalmente.

In [ ]:
future_vol = load_result('06_barrido_volatilidad_futura.csv')
halves = load_result('06b_estabilidad_mitades.csv')
display(
    future_vol.sort_values(['horizon', 'mutual_information'], ascending=[True, False])
    .groupby('horizon', as_index=False).head(3)
)
display(halves)
display(Image(filename=str(OUTPUT_DIR / 'fig_01_mi_lag_horizonte.png')))

## 10. Etapa H — regresión directa de volatilidad

Compara persistencia, regresión lineal y boosting con y sin `ΔBTC_y`. Si la mejora es prácticamente cero, la variable no debe venderse como regresor puntual aunque sí pueda servir como gate.

In [ ]:
reg_train = load_result('07b_regresion_train.csv')
reg_holdout = load_result('07_regresion_holdout.csv')
reg_walk_train = load_result('08b_regresion_walk_forward_train.csv')
reg_walk = load_result('08_regresion_walk_forward.csv')
print('TRAIN')
display(reg_train.sort_values('RMSE'))
print('HOLDOUT')
display(reg_holdout.sort_values('RMSE'))
if not reg_walk.empty:
    display(
        reg_walk.groupby(['model', 'feature_set'])[['MAE', 'RMSE', 'R2']]
        .agg(['mean', 'std'])
    )

## 11. Etapas I–J — clasificación de regímenes

Se comparan 3 regímenes, 5 regímenes con 4.5% en cada cola extrema y 5 quintiles libres. Macro-F1 y balanced accuracy son las métricas principales; accuracy sola puede ocultar el colapso de clases minoritarias.

In [ ]:
classification_train = load_result('09c_clasificacion_train.csv')
classification = load_result('09_clasificacion_holdout.csv')
print('TRAIN')
display(
    classification_train.sort_values(['regime_scheme', 'macro_f1'], ascending=[True, False])
    [['regime_scheme', 'model', 'feature_set', 'accuracy',
      'balanced_accuracy', 'macro_f1', 'log_loss']]
)
print('HOLDOUT')
display(
    classification.sort_values(['regime_scheme', 'macro_f1'], ascending=[True, False])
    [['regime_scheme', 'model', 'feature_set', 'accuracy',
      'balanced_accuracy', 'macro_f1', 'log_loss']]
)
display(Image(filename=str(OUTPUT_DIR / 'fig_02_macro_f1_holdout.png')))

## 12. Recall y F1 por régimen

Esta tabla revela si un promedio aparentemente bueno se obtuvo sin identificar los estados extremos.

In [ ]:
by_class = load_result('10_clasificacion_por_clase_holdout.csv')
display(
    by_class.sort_values(['regime_scheme', 'model', 'class_id'])
    [['regime_scheme', 'model', 'feature_set', 'class_name',
      'precision', 'recall', 'f1', 'support']]
)

## 13. Validación walk-forward

La media resume desempeño, pero el conteo de folds ganados muestra consistencia. Semillas y folds responden preguntas distintas: los folds prueban estabilidad temporal.

In [ ]:
walk_train = load_result('09d_clasificacion_walk_forward_train.csv')
walk = load_result('09b_clasificacion_walk_forward.csv')
if walk.empty:
    print('Walk-forward desactivado. Activá RUN_WALK_FORWARD y volvé a ejecutar el pipeline.')
else:
    walk_summary = (
        walk.groupby(['regime_scheme', 'model', 'feature_set'])
        [['balanced_accuracy', 'macro_f1', 'log_loss']]
        .agg(['mean', 'std'])
    )
    walk_train_summary = (
        walk_train.groupby(['regime_scheme', 'model', 'feature_set'])
        [['balanced_accuracy', 'macro_f1', 'log_loss']]
        .agg(['mean', 'std'])
    )
    print('TRAIN POR FOLD')
    display(walk_train_summary)
    print('VALIDACIÓN POR FOLD')
    display(walk_summary)

## 14. Importancia de variables

La importancia se agrega por familia. No debe interpretarse como causalidad ni como porcentaje de explicación porque existen variables correlacionadas.

In [ ]:
importance = load_result('11_importancia_features_holdout.csv')
family_importance = (
    importance.groupby(['regime_scheme', 'model', 'feature_set', 'feature_family'], as_index=False)
    ['importance_share'].sum()
    .sort_values(['regime_scheme', 'model', 'importance_share'], ascending=[True, True, False])
)
display(family_importance)
display(
    importance.sort_values('importance_share', ascending=False)
    [['regime_scheme', 'model', 'feature', 'feature_family',
      'importance_method', 'importance_share']].head(40)
)

## 15. Etapa K — utilidad incremental de la topología informacional

Esta es la conclusión central. Cada modelo con `BTC_y` se compara con un control de la misma arquitectura. Los cuatro `gain_*` están orientados de modo que un valor positivo favorece la topología.

In [ ]:
utility = load_result('13_utilidad_topologia.csv')
utility_summary = load_result('14_resumen_utilidad_topologia.csv')
display(utility.sort_values(['regime_scheme', 'contrast', 'split']))
display(utility_summary)
display(Image(filename=str(OUTPUT_DIR / 'fig_03_utilidad_incremental.png')))

## 16. Lectura científica automática

El bloque resume dirección, magnitud y estabilidad del aporte sin convertir una diferencia predictiva en afirmación causal.

In [ ]:
incremental = utility[utility['contrast'].str.startswith('incremental', na=False)].copy()
holdout_incremental = incremental[incremental['split'] == 'holdout']
walk_incremental = incremental[incremental['split'].str.startswith('fold_')]

print('HOLDOUT — utilidad incremental por esquema')
display(
    holdout_incremental[['regime_scheme', 'contrast', 'model', 'baseline_model',
                         'gain_balanced_accuracy', 'gain_macro_f1',
                         'gain_log_loss', 'relative_gain_macro_f1']]
)

if not walk_incremental.empty:
    stability = (
        walk_incremental.groupby(['regime_scheme', 'contrast'], as_index=False)
        .agg(
            folds=('split', 'count'),
            mean_gain_macro_f1=('gain_macro_f1', 'mean'),
            wins_macro_f1=('gain_macro_f1', lambda x: int((x > 0).sum())),
            mean_gain_log_loss=('gain_log_loss', 'mean'),
        )
    )
    print('WALK-FORWARD — estabilidad temporal')
    display(stability)

print(
    'Interpretación: hay evidencia favorable cuando el gain es positivo en holdout '
    'y se repite en la mayoría de los folds. Esto valida utilidad predictiva '
    'incremental de esta operacionalización de primer orden; no demuestra causalidad.'
)

## 17. Manifiesto reproducible

Conserva configuración, backend, ventanas, columnas, tamaños muestrales y contrato temporal de la corrida.

In [ ]:
manifest_path = OUTPUT_DIR / 'manifest.json'
with manifest_path.open(encoding='utf-8') as handle:
    manifest = json.load(handle)
display(manifest)

## 18. Prompts técnicos por modelo

Se genera un prompt autocontenido para cada estimador único. Los benchmarks también se documentan, aclarando que capas, activaciones, optimizador, batch y épocas no aplican. Las métricas se insertan desde los CSV de la corrida actual, no se escriben a mano.

In [ ]:
confusions = load_result('10b_matrices_confusion_holdout.csv')

def compact_table(frame, columns):
    if frame.empty:
        return 'No disponible en esta corrida.'
    existing = [column for column in columns if column in frame.columns]
    table = frame[existing].copy()
    numeric = table.select_dtypes(include='number').columns
    table[numeric] = table[numeric].round(6)
    return table.to_string(index=False)

def architecture_and_optimization(model, family):
    if model == 'persistence':
        return (
            'Benchmark determinista: proyecta sigma_past como sigma_future. No es una red; '
            'no tiene capas, dimensiones, activaciones, Dropout, BatchNorm ni L1/L2.',
            'No se entrena: no hay loss optimizada, optimizador, learning rate, batch size ni épocas.'
        )
    if model.startswith('linear_'):
        return (
            'Regresión lineal OLS de una sola salida, equivalente a una transformación afín '
            'con activación identidad. Sin capas ocultas, Dropout, BatchNorm ni regularización explícita.',
            'Mínimos cuadrados ordinarios de scikit-learn; error cuadrático como criterio implícito. '
            'No usa learning rate, weight decay, batch size ni épocas iterativas.'
        )
    if model.startswith('histgb_') and family == 'regression':
        return (
            'HistGradientBoostingRegressor: ensamble secuencial de árboles histogramados, '
            'max_depth=5 y max_leaf_nodes=15. No usa capas neuronales, activaciones, Dropout o BatchNorm.',
            'Loss squared_error; learning_rate=0.03; max_iter=300; random_state=42. '
            'No existe batch size y no se configuró penalización L1/L2 adicional.'
        )
    if model == 'dummy_most_frequent':
        return (
            'DummyClassifier determinista que siempre pronostica la clase más frecuente de train. '
            'No tiene red, capas, activaciones ni regularización.',
            'No optimiza parámetros: sin loss entrenada, optimizador, learning rate, batch o épocas.'
        )
    if model.startswith('tree_'):
        return (
            'DecisionTreeClassifier CART, max_depth=4 y min_samples_leaf=250. '
            'La profundidad y el mínimo por hoja actúan como regularización estructural; '
            'no hay capas neuronales, activaciones, Dropout ni BatchNorm.',
            'Criterio Gini de scikit-learn; crecimiento voraz del árbol; random_state=42. '
            'No usa optimizador por gradiente, learning rate, batch size ni épocas.'
        )
    if model.startswith('lightgbm_'):
        return (
            'LGBMClassifier multiclase: gradient boosting de 300 árboles, num_leaves=15 y '
            'max_depth=5. La restricción de hojas/profundidad regulariza la complejidad; '
            'no es una red neuronal y no usa Dropout o BatchNorm.',
            'Objective=multiclass con log-loss; learning_rate=0.03; n_estimators=300; '
            'random_state=42; sin L1/L2 explícitos. No existe batch size; cada árbol es una iteración.'
        )
    if model.startswith('histgb_'):
        return (
            'HistGradientBoostingClassifier multiclase: ensamble de árboles histogramados, '
            'max_depth=5 y max_leaf_nodes=15. No es una red neuronal; no usa Dropout o BatchNorm.',
            'Loss log_loss; learning_rate=0.03; max_iter=300; random_state=42. '
            'Sin batch size ni regularización L1/L2 explícita.'
        )
    return ('Arquitectura no reconocida automáticamente.', 'Optimización no reconocida automáticamente.')

def feature_context(feature_set, family):
    descriptions = {
        'sigma_past': 'volatilidad RMS pasada disponible en t',
        'sigma_past+dBTC_y': 'volatilidad pasada más variación contemporánea de la cuota BTC_y',
        'dummy': 'la entrada se ignora; el benchmark usa únicamente la frecuencia de clases de train',
        'simple_volatility': 'sigma_past',
        'simple_topology': 'BTC_y y dBTC_y',
        'simple_combined': 'sigma_past, BTC_y y dBTC_y',
        'advanced_volatility': 'estado multiescala causal de volatilidad: nivel, lags, medias, dispersión y EMA',
        'advanced_topology': 'estado multiescala causal de BTC_y: nivel, cambios, Shannon, logit, lags, rolling y EMA',
        'advanced_combined': 'familias multiescala de volatilidad y BTC_y más interacciones entre ambas',
    }
    task = (
        'regresión de sigma_future a 20 minutos'
        if family == 'regression'
        else 'clasificación temporal de 3 regímenes, 5 regímenes con colas 4.5% y 5 quintiles'
    )
    return (
        f'Problema: {task}. Features: {descriptions.get(feature_set, feature_set)}. '
        'Preprocesamiento: alineación cronológica, transformaciones causales, eliminación explícita '
        'de NaN/inf, holdout final del 30%, purga h=2, sin shuffle y umbrales ajustados sólo en train. '
        'Los árboles no requieren escalado ni codificación one-hot.'
    )

def confusion_text(model):
    selected = confusions[confusions['model'] == model]
    if selected.empty:
        return 'No aplica para regresión o no está disponible.'
    blocks = []
    for scheme, part in selected.groupby('regime_scheme', sort=False):
        matrix = part.pivot(index='true_class', columns='predicted_class', values='count').fillna(0).astype(int)
        blocks.append(f'{scheme}:\n{matrix.to_string()}')
    return '\n\n'.join(blocks)

def build_model_prompt(model, family, feature_set):
    architecture, optimization = architecture_and_optimization(model, family)
    if family == 'regression':
        train_metrics = reg_train[reg_train['model'] == model]
        test_metrics = reg_holdout[reg_holdout['model'] == model]
        wf_train = (reg_walk_train[reg_walk_train['model'] == model]
                    if not reg_walk_train.empty else pd.DataFrame())
        wf_eval = (reg_walk[reg_walk['model'] == model]
                   if not reg_walk.empty else pd.DataFrame())
        metric_columns = ['split', 'MAE', 'RMSE', 'R2', 'n_train', 'n_eval']
        per_class = 'No aplica: el target es continuo.'
        matrices = 'No aplica: es un modelo de regresión.'
    else:
        train_metrics = classification_train[classification_train['model'] == model]
        test_metrics = classification[classification['model'] == model]
        wf_train = (walk_train[walk_train['model'] == model]
                    if not walk_train.empty else pd.DataFrame())
        wf_eval = (walk[walk['model'] == model]
                   if not walk.empty else pd.DataFrame())
        metric_columns = ['regime_scheme', 'split', 'accuracy', 'balanced_accuracy',
                          'macro_f1', 'log_loss', 'n_train', 'n_eval']
        per_class = compact_table(
            by_class[by_class['model'] == model],
            ['regime_scheme', 'class_name', 'precision', 'recall', 'f1', 'support']
        )
        matrices = confusion_text(model)

    if wf_train.empty:
        wf_train_text = 'No disponible: walk-forward no ejecutado.'
        wf_eval_text = 'No disponible: walk-forward no ejecutado.'
    else:
        group_cols = ['regime_scheme'] if family == 'classification' else []
        numeric = (['accuracy', 'balanced_accuracy', 'macro_f1', 'log_loss']
                   if family == 'classification' else ['MAE', 'RMSE', 'R2'])
        if group_cols:
            wf_train_summary = wf_train.groupby(group_cols)[numeric].agg(['mean', 'std']).reset_index()
            wf_eval_summary = wf_eval.groupby(group_cols)[numeric].agg(['mean', 'std']).reset_index()
        else:
            wf_train_summary = wf_train[numeric].agg(['mean', 'std']).T.rename_axis('metric').reset_index()
            wf_eval_summary = wf_eval[numeric].agg(['mean', 'std']).T.rename_axis('metric').reset_index()
        wf_train_text = wf_train_summary.round(6).to_string(index=False)
        wf_eval_text = wf_eval_summary.round(6).to_string(index=False)

    return f'''Actuá como revisor metodológico de modelos predictivos para series temporales financieras.

Analizá el modelo `{model}` sin inventar componentes y usando exclusivamente la ficha y métricas siguientes.

1. ARQUITECTURA
{architecture}

2. OPTIMIZACIÓN
{optimization}

3. MÉTRICAS
Entrenamiento del ajuste final:
{compact_table(train_metrics, metric_columns)}

Holdout cronológico:
{compact_table(test_metrics, metric_columns)}

Train walk-forward, media y desvío:
{wf_train_text}

Validación walk-forward, media y desvío:
{wf_eval_text}

Métricas por clase en holdout:
{per_class}

Matrices de confusión holdout:
{matrices}

4. CONTEXTO DE LOS DATOS
{feature_context(feature_set, family)}

Tarea: redactá una ficha técnica rigurosa; compará train contra validación y test; diagnosticá
posible sobreajuste o subajuste; explicá el equilibrio entre clases; distinguí desempeño
predictivo de causalidad; y cerrá con fortalezas, límites y siguiente prueba recomendada.
Si un campo típico de redes neuronales no aplica, indicá explícitamente por qué.'''

regression_models = reg_holdout[['model', 'feature_set']].drop_duplicates().assign(family='regression')
classification_models = classification[['model', 'feature_set']].drop_duplicates().assign(family='classification')
model_catalog = pd.concat([regression_models, classification_models], ignore_index=True).drop_duplicates(['family', 'model'])
MODEL_PROMPTS = {
    f'{row.family}::{row.model}': build_model_prompt(row.model, row.family, row.feature_set)
    for row in model_catalog.itertuples(index=False)
}

prompt_document = ['# Prompts técnicos por modelo', '']
for number, (model, prompt) in enumerate(MODEL_PROMPTS.items(), start=1):
    prompt_document.extend([f'## {number}. {model}', '', '```text', prompt, '```', ''])
PROMPTS_PATH = OUTPUT_DIR / 'prompts_tecnicos_por_modelo.md'
PROMPTS_PATH.write_text('\n'.join(prompt_document), encoding='utf-8')

print(f'{len(MODEL_PROMPTS)} prompts guardados en: {PROMPTS_PATH}')
display(pd.DataFrame({'model': MODEL_PROMPTS.keys(), 'prompt_chars': [len(p) for p in MODEL_PROMPTS.values()]}))

## 19. Consultar un prompt individual

Reemplazá `MODEL_NAME` por cualquiera de los nombres de la tabla anterior.

In [ ]:
MODEL_NAME = next(iter(MODEL_PROMPTS))
display(Markdown(f'### Prompt: `{MODEL_NAME}`\n\n```text\n{MODEL_PROMPTS[MODEL_NAME]}\n```'))

## Conclusión de alcance

El experimento evalúa si una coordenada informacional del simplex aporta capacidad de identificación de regímenes. Sus variables son componentes NPP de primer orden: accesibles y transferibles, pero todavía sujetos a mejorar con microestructura, volatilidad condicional, liquidez efectiva y mediciones directas de resistencia sistémica.

El uso posterior propuesto es un gate probabilístico de cinco estados: `[P_XL, P_L, P_M, P_H, P_XH]`, no una predicción mecánica de precio.